In [ ]:
import pandas as pd

In [ ]:
claims = pd.read_csv('claims_and_billing.csv')
claims['claim_billing_date'] = pd.to_datetime(claims['claim_billing_date'], dayfirst=True)
claims['month'] = claims['claim_billing_date'].dt.strftime('%Y-%m')


In [ ]:
monthly = claims.groupby('month').agg(
    total_claims=('claim_id', 'count'),
    denied_claims=('claim_status', lambda x: (x == 'Denied').sum()),
    total_billed=('billed_amount', 'sum'),
    total_paid=('paid_amount', 'sum')
).reset_index()


In [ ]:
monthly['denial_rate'] = (monthly['denied_claims'] / monthly['total_claims']) * 100
monthly['net_collection_rate'] = (monthly['total_paid'] / monthly['total_billed']) * 100


In [ ]:
print(monthly.to_string(index=False))

  month  total_claims  denied_claims  total_billed  total_paid  denial_rate  net_collection_rate
2025-01          3527            357    5468105.54  3157415.08    10.121917            57.742395
2025-02         17464           1749   28469985.49 16639326.35    10.014888            58.445152
2025-03         20960           2125   33781199.32 19726806.78    10.138359            58.395815
2025-04         16467           1637   26747450.01 15694757.61     9.941094            58.677585
2025-05          1220            130    1936566.95  1130012.19    10.655738            58.351310


In [ ]:
# Payer breakdown
payer_kpis = claims.groupby('insurance_provider').agg(
    total_claims=('claim_id', 'count'),
    denial_rate=('claim_status', lambda x: (x == 'Denied').mean() * 100),
    billed=('billed_amount', 'sum'),
    paid=('paid_amount', 'sum')
    ).reset_index()
payer_kpis['ncr'] = (payer_kpis['paid'] / payer_kpis['billed']) * 100
print(payer_kpis.to_string(index=False))



insurance_provider  total_claims  denial_rate      billed        paid       ncr
             Aetna          8567     8.266693 15931881.91 10220394.03 64.150576
              BCBS          8516     8.336665 15937527.71 10319121.06 64.747314
             Cigna          8481     8.784670 16379391.57 10552126.52 64.423190
            Humana          8450     8.504635 15957291.04 10324831.81 64.702911
          Medicaid          8572     9.240206 16297387.90 10458464.11 64.172640
          Medicare          8460     8.344185 16068518.96 10437368.68 64.955387
               UHC          8592     8.497317 16328640.75 10533344.33 64.508397
